In [1]:
from pathlib import Path
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

/Users/surya/miniconda3/envs/clean311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
candidates = [
    Path('train_data'),
    Path('../train_data'),
    Path.cwd() / 'train_data',
    Path.cwd().parent / 'train_data',
]
train_data_dir = next((p.resolve() for p in candidates if p.exists()), None)
if train_data_dir is None:
    raise FileNotFoundError(
        "Could not find 'train_data' folder. Tried: " + ", ".join(str(p) for p in candidates)
    )
print('Using train_data:', train_data_dir)

seed = 42
TRAIN_FRAC = 0.90  # train uses only 90% of images

# Pretrained ImageNet backbones (including DenseNet121) expect ImageNet normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dataset = datasets.ImageFolder(root=str(train_data_dir), transform=transform)
num_classes = len(dataset.classes)
print('Num classes:', num_classes)

# Deterministic 90/10 split (disjoint)
n_all = len(dataset)
k_train = int(round(TRAIN_FRAC * n_all))
k_train = max(1, min(k_train, n_all - 1))  # ensure at least 1 train and 1 test

g = torch.Generator().manual_seed(seed)
perm = torch.randperm(n_all, generator=g)
train_idx = perm[:k_train].tolist()
test_idx = perm[k_train:].tolist()

train_img = Subset(dataset, train_idx)
test_img = Subset(dataset, test_idx)

batch_size = 32
num_workers = 4

g_loader = torch.Generator().manual_seed(seed)
train_loader = DataLoader(
    train_img,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    generator=g_loader,
 )
test_loader = DataLoader(
    test_img,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    generator=g_loader,
 )

print('Total images:', n_all)
print(f"Train images: {len(train_img)} ({len(train_img)/n_all:.0%})")
print(f"Test images:  {len(test_img)} ({len(test_img)/n_all:.0%})")
print('Classes (folder order used for labels):')
for i, name in enumerate(dataset.classes):
    print(f'  {i}: {name}')


Using train_data: /Users/surya/EDU/Ass 2/train_data
Num classes: 30
Total images: 6993
Train images: 6294 (90%)
Test images:  699 (10%)
Classes (folder order used for labels):
  0: Airport
  1: BareLand
  2: BaseballField
  3: Beach
  4: Bridge
  5: Center
  6: Church
  7: Commercial
  8: DenseResidential
  9: Desert
  10: Farmland
  11: Forest
  12: Industrial
  13: Meadow
  14: MediumResidential
  15: Mountain
  16: Park
  17: Parking
  18: Playground
  19: Pond
  20: Port
  21: RailwayStation
  22: Resort
  23: River
  24: School
  25: SparseResidential
  26: Square
  27: Stadium
  28: StorageTanks
  29: Viaduct


In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def set_seed(s: int) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss()

    loss_sum = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        loss_sum += float(loss.item()) * x.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())

    return loss_sum / max(1, total), correct / max(1, total)


def _make_subset_indices(train_pool_indices: list[int], frac: float, *, seed_: int) -> list[int]:
    # Deterministic subset selection with the assigned seed.
    # Uses a fixed permutation so 5% ⊆ 20% ⊆ 100%.
    g = torch.Generator().manual_seed(seed_)
    perm = torch.randperm(len(train_pool_indices), generator=g).tolist()
    k = int(round(frac * len(train_pool_indices)))
    k = max(1, min(k, len(train_pool_indices)))
    return [train_pool_indices[i] for i in perm[:k]]


class FrozenBackboneClassifier(nn.Module):
    """Frozen pretrained backbone + trainable linear head."""

    def __init__(self, backbone: nn.Module, n_classes: int, *, seed_: int):
        super().__init__()
        self.backbone = backbone
        self.n_features = int(getattr(backbone, "num_features"))

        set_seed(seed_)
        self.head = nn.Linear(self.n_features, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            feats = self.backbone(x)
        return self.head(feats)

    def train(self, mode: bool = True):
        self.head.train(mode)
        self.backbone.eval()
        return self


def build_frozen_backbone(model_name: str, *, seed_: int) -> nn.Module:
    set_seed(seed_)
    backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
    for p in backbone.parameters():
        p.requires_grad_(False)
    backbone.eval()
    backbone.to(device)
    return backbone


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    *,
    epochs: int,
    lr: float,
    weight_decay: float,
) -> None:
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()


# Backbones to compare (timm model names)
backbones: dict[str, str] = {
    "ResNet-50": "resnet50",
    "DenseNet-121": "densenet121",
    "EfficientNet-B0": "efficientnet_b0",
}

regimes = [1.0, 0.20, 0.05]

EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

batch_size = 32
num_workers = 0  # safer default on macOS notebooks

fewshot_results: dict[str, dict[float, dict[str, float]]] = {}

for backbone_label, model_name in backbones.items():
    # Backbone-specific transforms
    tmp = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
    cfg = resolve_data_config({}, model=tmp)
    train_tf = create_transform(**cfg, is_training=True)
    eval_tf = create_transform(**cfg, is_training=False)
    del tmp

    # Datasets share the same file ordering, but use different transforms
    train_ds = datasets.ImageFolder(root=str(train_data_dir), transform=train_tf)
    eval_ds = datasets.ImageFolder(root=str(train_data_dir), transform=eval_tf)

    # Validation is a deterministic subset of train_data
    val_set = Subset(eval_ds, val_idx)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    fewshot_results[backbone_label] = {}
    backbone = build_frozen_backbone(model_name, seed_=seed)

    for frac in regimes:
        subset_idx = _make_subset_indices(train_idx, frac, seed_=seed)

        train_set = Subset(train_ds, subset_idx)
        train_loader = DataLoader(
            train_set,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            generator=torch.Generator().manual_seed(seed),
        )

        train_eval_set = Subset(eval_ds, subset_idx)
        train_eval_loader = DataLoader(train_eval_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        model = FrozenBackboneClassifier(backbone, num_classes, seed_=seed).to(device)
        train_model(model, train_loader, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY)

        _, train_acc = evaluate(model, train_eval_loader)
        _, val_acc = evaluate(model, val_loader)

        fewshot_results[backbone_label][frac] = {
            "train_acc": float(train_acc),
            "val_acc": float(val_acc),
        }

In [13]:
import pandas as pd

regimes = [1.0, 0.20, 0.05]

rows: list[dict[str, object]] = []
for backbone_label, by_frac in fewshot_results.items():
    acc100 = float(by_frac[1.0]["val_acc"])
    acc5 = float(by_frac[0.05]["val_acc"])
    rel_drop = (acc100 - acc5) / max(1e-12, acc100)

    row = {
        "Backbone": backbone_label,
        "Val Acc (100%)": acc100,
        "Val Acc (20%)": float(by_frac[0.20]["val_acc"]),
        "Val Acc (5%)": acc5,
        "Δ = (Acc100-Acc5)/Acc100": rel_drop,
        "Gap 100% (train-val)": float(by_frac[1.0]["train_acc"]) - acc100,
        "Gap 20% (train-val)": float(by_frac[0.20]["train_acc"]) - float(by_frac[0.20]["val_acc"]),
        "Gap 5% (train-val)": float(by_frac[0.05]["train_acc"]) - acc5,
    }
    rows.append(row)

df = pd.DataFrame(rows)

# Display
fmt = {
    "Val Acc (100%)": "{:.4f}",
    "Val Acc (20%)": "{:.4f}",
    "Val Acc (5%)": "{:.4f}",
    "Δ = (Acc100-Acc5)/Acc100": "{:.4f}",
    "Gap 100% (train-val)": "{:+.4f}",
    "Gap 20% (train-val)": "{:+.4f}",
    "Gap 5% (train-val)": "{:+.4f}",
}

display(df.style.format(fmt))


,Backbone,Val Acc (100%),Val Acc (20%),Val Acc (5%),Δ = (Acc100-Acc5)/Acc100,Gap 100% (train-val),Gap 20% (train-val),Gap 5% (train-val)
0,ResNet-50,0.8941,0.8526,0.6624,0.2592,+0.0035,+0.0001,+0.1262
1,DenseNet-121,0.9642,0.9442,0.8240,0.1454,-0.0064,+0.0000,+0.1017
2,EfficientNet-B0,0.9499,0.9270,0.8183,0.1386,-0.0011,+0.0029,+0.1217
